![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 07: Geospatial & Spatial Epidemiology
***
**Learning objectives**
- Compute travel distance and catchment-area access metrics
- Implement the Two-Step Floating Catchment Area (2SFCA) method
- Build provider-to-population ratio metrics by service area
- Delineate Health Professional Shortage Areas (HPSA-style analysis)
- Compute network accessibility indices for hospital and primary care
- Map spatial disparities in healthcare access

**Estimated time:** 2.5 hours | **Level:** Advanced  
**Libraries:** `numpy`, `scipy`, `pandas`, `matplotlib`


## 1. Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
from scipy.ndimage import gaussian_filter; from scipy.spatial import cKDTree; from scipy import stats
warnings.filterwarnings("ignore"); import os; os.makedirs("/tmp/mod07", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)
NROW,NCOL=10,20; N=NROW*NCOL
row_idx=np.repeat(np.arange(NROW),NCOL); col_idx=np.tile(np.arange(NCOL),NROW)
lon=-120+col_idx*2.5+np.random.normal(0,0.3,N)
lat=28+row_idx*2.0+np.random.normal(0,0.2,N)
pct_poverty=10+8*np.random.beta(2,5,N)+3*np.sin(lon/20)
pct_uninsured=8+6*np.random.beta(2,4,N)+0.3*pct_poverty
pm25=8+4*np.random.gamma(2,1,N); population=np.random.lognormal(10.5,1.0,N).astype(int)
def sp(vals,sigma=2):
    grid=np.zeros((NROW,NCOL))
    for r,c,v in zip(row_idx,col_idx,vals): grid[r,c]=v
    return gaussian_filter(grid,sigma=sigma)[row_idx,col_idx]
spatial_risk=sp(np.random.normal(0,1,N))
cvd_rate=(180+1.2*pct_poverty+0.8*pct_uninsured+0.5*pm25+15*spatial_risk+np.random.normal(0,12,N))
expected=(cvd_rate/100000)*population*5; observed=np.random.poisson(expected).astype(int)
smr=observed/expected.clip(1)
df=pd.DataFrame({"county_id":[f"C{i:03d}" for i in range(N)],
    "lon":lon,"lat":lat,"row":row_idx,"col":col_idx,
    "pct_poverty":pct_poverty,"pct_uninsured":pct_uninsured,"pm25":pm25,
    "population":population,"cvd_rate":cvd_rate,
    "observed":observed,"expected":expected,"smr":smr})
# Simulate hospital locations
np.random.seed(5); N_HOSP=40
hosp_lon=np.random.uniform(lon.min(),lon.max(),N_HOSP)
hosp_lat=np.random.uniform(lat.min(),lat.max(),N_HOSP)
hosp_type=np.random.choice(["Critical_Access","Community","Regional","Academic"],N_HOSP,
                             p=[0.30,0.40,0.20,0.10])
hosp_beds=np.where(hosp_type=="Academic",np.random.randint(400,800,N_HOSP),
          np.where(hosp_type=="Regional",np.random.randint(150,400,N_HOSP),
          np.where(hosp_type=="Community",np.random.randint(50,200,N_HOSP),
                   np.random.randint(25,75,N_HOSP))))
df_hosp=pd.DataFrame({"lon":hosp_lon,"lat":hosp_lat,"type":hosp_type,"beds":hosp_beds})
print(f"N={N} counties | {N_HOSP} hospitals | CVD={cvd_rate.mean():.1f}/100k")

## 2. Access Metrics Framework

**Primary care access metrics:**

| Metric | Formula | Limitation |
|--------|---------|-----------|
| **Provider:population ratio** | Providers / Population | Ignores distance |
| **Distance to nearest** | min(d_ij) | Ignores capacity |
| **2SFCA** | Sum of weighted provider ratios | Most comprehensive |
| **Gravity model** | Sum(S_j / d^p) | Power-law decay |
| **Floating catchment** | Within d miles of provider | Binary, ignores gradients |

**Two-Step Floating Catchment Area (2SFCA):**
1. Step 1: For each provider j, compute R_j = Beds_j / Σ(Pop_i for i within d)
2. Step 2: For each county i, A_i = Σ(R_j for j within d of i)
Higher A_i = better access.


In [ ]:
from scipy.spatial import cKDTree

county_coords = np.column_stack([df.lon, df.lat])
hosp_coords   = np.column_stack([df_hosp.lon, df_hosp.lat])
tree_hosp     = cKDTree(hosp_coords)

# Distance from each county to each hospital
from scipy.spatial.distance import cdist
dist_matrix = cdist(county_coords, hosp_coords)  # (N_counties, N_hospitals)

# Catchment radius (approximate 60 miles in degree units ~0.87 degrees/mile)
CATCHMENT_DEG = 4.5  # ~5 degrees ≈ 300 miles

# Simple provider-to-population ratio (within catchment)
ppr = np.zeros(N)  # beds per 1000 population
for i in range(N):
    in_catchment = dist_matrix[i] <= CATCHMENT_DEG
    if in_catchment.any():
        total_beds = df_hosp.loc[in_catchment, "beds"].sum()
        ppr[i] = total_beds / (df.population.iloc[i] / 1000)
df["ppr_beds_per_1000"] = ppr

# Distance to nearest hospital
dist_nearest = dist_matrix.min(axis=1)
df["dist_nearest_hosp"] = dist_nearest

# 2SFCA Method
def two_sfca(county_pop, county_coords, provider_cap, provider_coords, catchment, decay_power=1):
    """Two-Step Floating Catchment Area method."""
    dist = cdist(county_coords, provider_coords)
    # Step 1: Compute provider-to-demand ratio
    R_j = np.zeros(len(provider_cap))
    for j in range(len(provider_cap)):
        in_zone = dist[:, j] <= catchment
        denom = (county_pop[in_zone] / (dist[in_zone, j] + 0.1)**decay_power).sum()
        R_j[j] = provider_cap[j] / denom if denom > 0 else 0
    # Step 2: Sum provider ratios accessible to each county
    A_i = np.zeros(len(county_pop))
    for i in range(len(county_pop)):
        in_zone = dist[i, :] <= catchment
        if in_zone.any():
            weights = 1 / (dist[i, in_zone] + 0.1)**decay_power
            A_i[i] = (R_j[in_zone] * weights).sum()
    return A_i

df["access_2sfca"] = two_sfca(
    df.population.values, county_coords,
    df_hosp.beds.values, hosp_coords,
    catchment=CATCHMENT_DEG, decay_power=2
)
print("Access metrics computed:")
print(f"  Beds per 1000 (simple PPR): mean={df.ppr_beds_per_1000.mean():.2f} SD={df.ppr_beds_per_1000.std():.2f}")
print(f"  Distance to nearest hospital: mean={df.dist_nearest_hosp.mean():.2f}° SD={df.dist_nearest_hosp.std():.2f}°")
print(f"  2SFCA access score: mean={df.access_2sfca.mean():.4f} SD={df.access_2sfca.std():.4f}")


In [ ]:
# Map access metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
ax = axes.flatten()

for i, (col, title, cmap, invert) in enumerate([
    ("ppr_beds_per_1000",  "Beds per 1,000 Pop (Simple PPR)", "Blues", False),
    ("dist_nearest_hosp",  "Distance to Nearest Hospital (°)", "RdYlGn_r", False),
    ("access_2sfca",       "2SFCA Access Score", "RdYlGn", False),
    ("pct_uninsured",      "% Uninsured (demand-side barrier)", "YlOrRd", False),
]):
    vals = df[col]
    sc = ax[i].scatter(df.lon, df.lat, c=vals, cmap=cmap, s=150, alpha=0.85,
                       edgecolors="white", linewidth=0.2)
    ax[i].scatter(df_hosp.lon, df_hosp.lat, marker="P", c="black", s=60, zorder=5, label="Hospital")
    plt.colorbar(sc, ax=ax[i]); ax[i].set_title(title, fontsize=10)
    ax[i].set_xlabel("Lon"); ax[i].set_ylabel("Lat")
    if i == 0: ax[i].legend(fontsize=8)
plt.suptitle("Healthcare Access Atlas: Multiple Metrics", fontsize=13)
plt.tight_layout(); plt.savefig("/tmp/mod07/access_metrics.png", bbox_inches="tight"); plt.show()


## 3. HPSA-Style Shortage Area Delineation

In [ ]:
# Federal HPSA criteria (simplified):
# Primary care shortage if provider:population ratio > 1:3,500
# or distance to nearest > 30 miles
# Medically underserved if: 3+ of 4 criteria met

HPSA_PPR_THRESHOLD = 0.286  # 1 per 3,500 = 0.286 per 1,000
HPSA_DIST_THRESHOLD = 3.5   # ~30 miles in degrees

# Score components (0-1 each)
# 1. Low beds per 1000 (inverted: lower = worse)
score_ppr  = 1 - (df.ppr_beds_per_1000 / df.ppr_beds_per_1000.quantile(0.9)).clip(0,1)
# 2. High distance to nearest hospital
score_dist = (df.dist_nearest_hosp / df.dist_nearest_hosp.quantile(0.9)).clip(0,1)
# 3. Low 2SFCA access
score_2sfca = 1 - (df.access_2sfca / df.access_2sfca.quantile(0.9)).clip(0,1)
# 4. High uninsured rate
score_uninsured = (df.pct_uninsured / df.pct_uninsured.quantile(0.9)).clip(0,1)

df["hpsa_score"]  = (score_ppr + score_dist + score_2sfca + score_uninsured) / 4
df["hpsa_status"] = pd.cut(df.hpsa_score,
                             bins=[0, 0.25, 0.50, 0.75, 1.01],
                             labels=["Adequate","Low shortage","Moderate shortage","HPSA"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
hpsa_colors = {"Adequate":"#4878CF","Low shortage":"#B3CDE3","Moderate shortage":"#FEC44F","HPSA":"#D65F5F"}
colors_hpsa = [hpsa_colors[s] for s in df.hpsa_status]
axes[0].scatter(df.lon, df.lat, c=colors_hpsa, s=180, edgecolors="white", linewidth=0.2)
axes[0].scatter(df_hosp.lon, df_hosp.lat, marker="P", c="black", s=70, zorder=5, label="Hospital")
axes[0].set_title("Health Professional Shortage Areas (HPSA-style)"); axes[0].legend(fontsize=9)
for label, color in hpsa_colors.items():
    n = (df.hpsa_status==label).sum()
    axes[0].plot([],[],"-",color=color,lw=8,label=f"{label} (n={n})")
axes[0].legend(fontsize=8,loc="lower right")
axes[0].set_xlabel("Lon"); axes[0].set_ylabel("Lat")

# Population in shortage areas
pop_by_status = df.groupby("hpsa_status",observed=True)["population"].sum()
pct_pop = pop_by_status / df.population.sum() * 100
axes[1].bar(pct_pop.index, pct_pop.values,
            color=[hpsa_colors[s] for s in pct_pop.index], alpha=0.85, edgecolor="white")
axes[1].set_ylabel("% of total population"); axes[1].set_title("Population by Access Category")
for i,(cat,val) in enumerate(pct_pop.items()):
    axes[1].text(i, val+0.3, f"{val:.1f}%", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod07/hpsa_analysis.png", bbox_inches="tight"); plt.show()

print("HPSA Analysis Summary:")
for status in ["Adequate","Low shortage","Moderate shortage","HPSA"]:
    n_c  = (df.hpsa_status==status).sum()
    pop_s = df.loc[df.hpsa_status==status,"population"].sum()
    pct  = pop_s/df.population.sum()*100
    print(f"  {status:22s}: {n_c:3d} counties | {pop_s:,} residents ({pct:.1f}%)")


## 4. Disparities in Access by SES

In [ ]:
# Do poorer counties have worse access?
from scipy.stats import pearsonr, spearmanr

access_corrs = {}
for access_metric, higher_is_better in [("ppr_beds_per_1000",True),
                                          ("dist_nearest_hosp",False),
                                          ("access_2sfca",True)]:
    r_poverty, p_pov = pearsonr(df.pct_poverty, df[access_metric])
    r_uniins, p_uni  = pearsonr(df.pct_uninsured, df[access_metric])
    if not higher_is_better:
        r_poverty, r_uniins = r_poverty, r_uniins  # for distance, positive r = worse access
    access_corrs[access_metric] = {"r_poverty":r_poverty,"p_poverty":p_pov,
                                    "r_uninsured":r_uniins,"p_uninsured":p_uni}

print("Access-Deprivation Correlations:")
print(f"{'Metric':25s} {'r(Poverty)':>12s} {'p':>8s} {'r(Uninsured)':>14s} {'p':>8s}")
print("-"*72)
for metric, vals in access_corrs.items():
    print(f"  {metric:23s} {vals['r_poverty']:>12.3f} {vals['p_poverty']:>8.4f} "
          f"{vals['r_uninsured']:>14.3f} {vals['p_uninsured']:>8.4f}")

# Concentration index (health equity measure)
def concentration_index(health, rank_var):
    n = len(health)
    rank = pd.Series(rank_var).rank() / n
    health_mean = health.mean()
    CI = 2 * np.cov(rank, health)[0,1] / health_mean
    return CI

ci_poverty_access = concentration_index(df.access_2sfca, df.pct_poverty)
print(f"\nConcentration Index (2SFCA access by poverty rank): {ci_poverty_access:.4f}")
print(f"  CI < 0 means better access is concentrated in RICHER areas (pro-rich inequality)")
print(f"  CI > 0 means better access is concentrated in POORER areas (pro-poor inequality)")
print(f"  CI = 0 means equal access distribution")


## 5. Knowledge Check
1. 2SFCA gives county A score=0.82 and county B score=0.14. Both have the same distance to the nearest hospital. What could explain the large difference?
2. The concentration index is -0.31. A health minister says "access is distributed equitably." Is this correct?
3. Your HPSA analysis uses distance in degrees. Why is this problematic, and what would you use instead?
4. A large Academic Medical Centre opens in a rural county. How would each of the four access metrics change?
5. Implement a Gini coefficient for healthcare access to quantify inequality across all counties.


In [ ]:
# Exercise 5 — Gini coefficient for access
def gini_coefficient(values):
    sorted_v = np.sort(values)
    n = len(sorted_v)
    cumulative = sorted_v.cumsum()
    total = cumulative[-1]
    lorenz = cumulative / total
    lorenz = np.insert(lorenz, 0, 0)
    x = np.linspace(0, 1, n+1)
    gini = 1 - 2 * np.trapz(lorenz, x)
    return gini, lorenz, x

for metric, label in [("access_2sfca","2SFCA Access"),("ppr_beds_per_1000","Beds per 1000")]:
    g, lorenz, x = gini_coefficient(df[metric].values)
    print(f"Gini coefficient for {label}: {g:.4f}")

g_access, lorenz_access, x_lor = gini_coefficient(df["access_2sfca"].values)
fig, ax = plt.subplots(figsize=(7,5))
ax.plot(x_lor, lorenz_access, "-", color="#D65F5F", lw=2.5, label=f"2SFCA Lorenz (Gini={g_access:.3f})")
ax.plot([0,1],[0,1],"k--",lw=1.5,label="Perfect equality")
ax.fill_between(x_lor, lorenz_access, x_lor, alpha=0.15, color="#D65F5F")
ax.set_xlabel("Cumulative % of counties (sorted by access)")
ax.set_ylabel("Cumulative % of total access")
ax.set_title("Lorenz Curve: Healthcare Access Distribution")
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod07/lorenz_access.png",bbox_inches="tight"); plt.show()


***
**Next → NB-08: Capstone — Spatial Health Inequity Atlas**
